# 第 03 章 · 模型通信：请求、流式响应与重试

模型响应不是一个瞬间出现的 JSON：网络会切碎 UTF-8 字符、参数会分多次到达，终态还可能缺失。
传输层负责还原消息，适配器负责确认完成，Dispatcher 才拥有工具执行权。
若每收到一个参数片段就执行，网络重试会把不完整输入甚至同一副作用交付多次。

我们用离线字节流验证这些边界。线协议刻意缩成 **NDJSON**，不是 SSE 或 Responses API 的实现；
它保留字节分帧、item 完成和 response 终态的差别。为了降低首次实现难度，本练习要求整响应完成才释放调用。
Codex 会在完整 `OutputItemDone` 时交接工具，不统一等待整个响应完成；最后会给出源码对照。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
transport = load_example('03', 'streaming')
import json

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 改变网络分片，完整参数应保持不变


    JSON 解析必须发生在完整帧上，UTF-8 解码必须记住前一片未结束的字节。
    参数片段只写入组装缓冲，`item_done` 才验证完整 JSON 对象。
    调整 `chunk_width` 到 1，可以稳定制造汉字被拆成多个字节的情况。


In [3]:
chunk_width = 1
arguments = {'path': '任务清单.py'}
raw_arguments = json.dumps(arguments, ensure_ascii=False)
split_at = len(raw_arguments) // 2
events = [
    {'type': 'delta', 'call_id': 'c1', 'text': raw_arguments[:split_at]},
    {'type': 'delta', 'call_id': 'c1', 'text': raw_arguments[split_at:]},
    {'type': 'item_done', 'call_id': 'c1', 'name': 'read_file'},
    {'type': 'completed', 'usage': {'input_tokens': 20, 'output_tokens': 8}},
]
payload = transport.wire(events)
result = transport.consume(transport.chunks(payload, chunk_width))
assert result['status'] == 'completed'
assert result['calls'][0]['arguments'] == arguments
assert result['usage']['presence'] == 'object'
{'byte_chunks': len(transport.chunks(payload, chunk_width)), 'decoded_call': result['calls'][0]}


{'byte_chunks': 258,
 'decoded_call': {'call_id': 'c1',
  'name': 'read_file',
  'arguments': {'path': '任务清单.py'}}}

## 2. 半个参数和半个响应，都不能冒充完成


    `item_done` 表示该 item 完整；`completed` 表示响应结束；任务完成还需要 Loop 的独立判断。
    在本教学适配器中，任一层缺失都不会交给工具。注意完整 item 已经到达、最后终态丢失时，
    `ready_count` 可以为 1，但 `calls` 仍为空，这是有意选择的保守策略。


In [4]:
half_argument = transport.consume(transport.chunks(transport.wire([
    {'type': 'delta', 'call_id': 'x', 'text': '{"path":'},
    {'type': 'item_done', 'call_id': 'x', 'name': 'read_file'},
]), 7))
no_terminal = transport.consume(transport.chunks(transport.wire(events[:-1]), 7))
cut_bytes = 4
partial_frame = transport.consume([payload[:-cut_bytes]])
assert half_argument['status'] == 'protocol_error' and half_argument['calls'] == []
assert no_terminal['status'] == 'missing_terminal' and no_terminal['ready_count'] == 1
assert no_terminal['calls'] == []
assert partial_frame['status'] == 'partial_frame' and partial_frame['calls'] == []
{'bad_arguments': half_argument['status'], 'lost_terminal': no_terminal['status'], 'cut_frame': partial_frame['status']}


{'bad_arguments': 'protocol_error',
 'lost_terminal': 'missing_terminal',
 'cut_frame': 'partial_frame'}

## 3. 不完整结果、取消和 usage 独立记录


    token 用量描述服务计算过什么，不证明工具或任务成功。失败响应也可能带 usage；缺少字段与显式 null
    是不同观察。取消本地读取同样不能证明服务端没有算过。本例只取消本地字节消费，不声称撤回真实服务请求。


In [5]:
failure_events = events[:-1] + [{'type': 'incomplete', 'usage': {'output_tokens': 8}}]
failure = transport.consume([transport.wire(failure_events)])
missing = transport.consume([transport.wire([{'type': 'completed'}])])
explicit_null = transport.consume([transport.wire([{'type': 'completed', 'usage': None}])])
cancel_after_chunks = 2
cancelled = transport.consume(transport.chunks(payload, 10), cancel_after=cancel_after_chunks)
assert failure['status'] == 'incomplete' and failure['calls'] == []
assert failure['usage']['raw']['output_tokens'] == 8
assert missing['usage']['presence'] == 'missing'
assert explicit_null['usage']['presence'] == 'null'
assert cancelled['status'] == 'cancelled' and cancelled['calls'] == []
{'incomplete_usage': failure['usage'], 'usage_absence': [missing['usage'], explicit_null['usage']], 'cancelled': cancelled['status']}


{'incomplete_usage': {'presence': 'object', 'raw': {'output_tokens': 8}},
 'usage_absence': [{'presence': 'missing', 'raw': None},
  {'presence': 'null', 'raw': None}],
 'cancelled': 'cancelled'}

## 4. 重试保留逻辑请求，改变尝试身份


    一次逻辑采样可以经历两次传输尝试。输入快照应固定，attempt ID 必须不同，方便分辨同一请求的重试。
    此处第一次流缺失终态，且尚未释放工具，因此允许再读一次受控响应；`incomplete` 和取消不在自动重试名单。
    把 `retry_budget` 设为 0，可观察原失败直接返回。没有真实 HTTP、退避、WebSocket 或服务缓存实验。


In [6]:
retry_budget = 1
request = {'request_id': 'logical-1', 'model': 'fixture', 'messages': ['修复筛选']}
report = transport.sample_fixture(request, [[transport.wire(events[:-1])], [payload]], max_retries=retry_budget)
assert len({a['request_digest'] for a in report['attempts']}) == 1
assert len({a['attempt_id'] for a in report['attempts']}) == len(report['attempts'])
assert report['result']['status'] == ('completed' if retry_budget >= 1 else 'missing_terminal')
stopped = transport.sample_fixture(request, [[transport.wire(failure_events)], [payload]], max_retries=3)
assert len(stopped['attempts']) == 1 and stopped['result']['calls'] == []
report


{'result': {'status': 'completed',
  'calls': [{'call_id': 'c1',
    'name': 'read_file',
    'arguments': {'path': '任务清单.py'}}],
  'ready_count': 1,
  'terminal': 'completed',
  'usage': {'presence': 'object',
   'raw': {'input_tokens': 20, 'output_tokens': 8}}},
 'attempts': [{'attempt_id': 'logical-1:a1',
   'request_digest': '6815faed9c924f7d0006443dcbfb927c4afddac9be9caee5d7d2a6d4856e9476',
   'outcome': 'missing_terminal'},
  {'attempt_id': 'logical-1:a2',
   'request_digest': '6815faed9c924f7d0006443dcbfb927c4afddac9be9caee5d7d2a6d4856e9476',
   'outcome': 'completed'}]}

## 5. 重复 item 不能生成第二次工具交付


    网络重放可能重复同一 `item_done`。这里拒绝整响应，使调用方处理协议异常。
    这只防当前适配器的重复组装；真正的副作用去重要由下一章 Dispatcher 的调用身份登记承担。


In [7]:
duplicate_events = events[:-1] + [events[2], events[-1]]
duplicate = transport.consume([transport.wire(duplicate_events)])
assert duplicate['status'] == 'protocol_error'
assert duplicate['calls'] == []
assert 'duplicate_item_done' in duplicate['error']
duplicate


{'status': 'protocol_error',
 'calls': [],
 'ready_count': 1,
 'terminal': None,
 'usage': {'presence': 'missing', 'raw': None},
 'error': 'duplicate_item_done'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [ResponseEvent](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/common.rs#L117-L154)：ResponseEvent 明确区分 OutputItemDone、OutputItemAdded、OutputTextDelta 与 Completed；Completed 仍有可选 end_turn。

- [process_sse_with_treatment](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/sse/responses.rs#L620-L655)：SSE EOF 且未接受终态时返回 Stream 错误；idle timeout 也是 Stream 错误；只有已解析 Completed 才结束正常读流。

- [try_run_sampling_request](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn.rs#L3201-L3220)：核心处理 OutputItemDone 时建立 HandleOutputCtx，调用 handle_output_item_done，并把返回的 tool_future 加入 in_flight；没有统一等待 ResponseEvent::Completed 才交接 item。

- [process_responses_event](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/sse/responses.rs#L471-L540)：response.incomplete: content_filter映射ContentFilter；reason不是interrupted则返回ResponseIncomplete（缺失reason用unknown）；interrupted继续按Completed解析并强制end_turn=false。核心遇end_turn=false设needs_follow_up。 response.failed另经parse_failed_response分类为API错误。


## 现在应能解释什么


    能分别指出字节帧、参数对象、item、response 与任务的完成边界；能解释为什么 usage 与成功分开，
    以及逻辑请求 ID 为什么不能替代 attempt ID、服务请求 ID 或响应 ID。

    接回 Loop 时，Provider 输出明确终态和完整工具调用；传输观察者只有展示权。
    本章仅证明离线组装与重试决策，不证明真实 API、SSE/WebSocket 互通或任何模型效果。
